## 配置环境

In [9]:
from agent.llm_client import LLMClient
from config import AgentConfig

config = AgentConfig()
llm = LLMClient()

## 设计工具
这里我想通过三个工具简单实现穿衣推荐的功能，穿衣和天气密切相关，所以精确的穿衣 -> 获得精确的地址\时间\天气
- date_tool 用来获取指定/当前日期
- address_tool 用来获取目的地/当前位置
- weather_tool 用来获取天气

In [10]:
from tools.registry import ToolRegistry
from tools.address_tool import AddressTool
from tools.weather_tool import WeatherTool
from tools.date_tool import DateTool

weather_tool = WeatherTool()
address_tool = AddressTool()
date_tool = DateTool()

### 三个工具都配备
tools_all = ToolRegistry()
tools_all.register(weather_tool)  
tools_all.register(address_tool)   
tools_all.register(date_tool) 

### 没有天气工具
tools_part_1 = ToolRegistry()
tools_part_1.register(date_tool)  
tools_part_1.register(address_tool)   

### 没有任何工具
tools_none = ToolRegistry()

In [11]:
from agent.react_agent import ReActAgent

agent_all = ReActAgent(
    llm=llm,
    tool_registry=tools_all,
    config=config,
)

agent_part_1 = ReActAgent(
    llm=llm,
    tool_registry=tools_part_1,
    config=config,
)

agent_none = ReActAgent(
    llm=llm,
    tool_registry=tools_none,
    config=config,
)

### 问题

In [12]:
question = input("Question: ").strip()

if not question:
    print("问题不能为空")
else:
    print(f"问题: {question}")

问题: 明天北京会下雨吗？


### 三个工具

In [13]:
result = agent_all.run(question)

Step 1/6
Thought:我需要先获取明天的日期，然后查询北京那天的天气来判断是否会下雨。
Action:date_tool
Action Input:{'date': '明天'}
Observation:date=2026-07-16, weekday=Thursday
Tool Status:success
Tool Error:None

Step 2/6
debug
Step 3/6
Thought:None
Action:weather_tool
Action Input:{'location': '北京', 'date': '2026-07-16'}
Observation:解析成功
城市: Beijing
州/省: Beijing
国家: CN
坐标: (39.9057, 116.3913)

2026-07-16 天气
最高温度: 33.2°C
最低温度: 25.4°C
天气状况: 多云
Tool Status:success
Tool Error:None

Step 4/6
Thought:我已经获取了明天（2026-07-16）北京的天气状况为“多云”，并非降雨天气，因此明天北京不会下雨。
Final Answer:明天（2026-07-16）北京的天气为多云，最高温度33.2°C，最低温度25.4°C，不会下雨。



### 两个工具

In [14]:
result = agent_part_1.run(question)

Step 1/6
Thought:用户询问明天北京是否会下雨，我需要知道明天的日期以及北京的天气情况。我可以使用date_tool获取明天的日期，但天气信息需要天气查询工具，而当前可用的工具只有date_tool和address_tool，没有提供weather_tool，因此无法获取天气预报数据。
Final Answer:由于当前可用的工具中没有天气查询功能，我无法获取明天的天气预报信息，因此无法判断明天北京是否会下雨。请提供天气查询工具或自行查询。



### 没有工具

In [15]:
result = agent_none.run(question)

Step 1/6
Thought:用户想知道明天北京是否会下雨，我需要先知道明天的具体日期。
Action:date_tool
Action Input:{'date': '明天'}
Observation:Tool 'date_tool' does not exist.
Tool Status:tool_not_found
Tool Error:tool_not_found

Step 2/6
Thought:工具 `date_tool` 似乎不存在，无法获取明天的具体日期。我需要重试或尝试其他方法。也许可以尝试获取今天日期，然后手动计算明天，或者使用不同的参数。
Action:date_tool
Action Input:{'date': 'today'}
Observation:Tool 'date_tool' does not exist.
Tool Status:tool_not_found
Tool Error:tool_not_found

Step 3/6
debug
Step 4/6
Thought:当前没有可用的工具来获取明天的日期和北京的天气信息，因此无法回答该问题。
Final Answer:无法获取信息。

